In [ ]:
%run 10_MNESIS_polychronous-chains.ipynb

## optimizing learning parameters with optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning,)
optuna_filename = data_cache / f"{opt.datetag}_optuna.sqlite3" 
# if True : 
if RECOMPUTE : 
    optuna_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
# optuna_filename.unlink(missing_ok=True) # FORCING RECOMPUTE    
%ls -l {optuna_filename}

In [ ]:
def objective(trial):    
    new_opt = Params(num_epochs=200)

    new_opt.do_deconv = trial.suggest_categorical('do_deconv', [True, False])
    # new_opt.learn_beta = trial.suggest_categorical('learn_beta', [True, False])
    # new_opt.learn_threshold = trial.suggest_categorical('learn_threshold', [True, False])
    new_opt.reset_mechanism = trial.suggest_categorical('reset_mechanism', ['zero', 'subtract'])
    new_opt.optimizer = trial.suggest_categorical('optimizer', ['adamw', 'rmsprop', 'adadelta', 'sgd', 'adam']) # 'adagrad', 'sparseadam', 

    new_opt.surrogate_name = trial.suggest_categorical('surrogate_name', ['FastSigmoid', 'LeakySpikeOperator', 'ATan', 'Sigmoid', 'SpikeRateEscape'])
    new_opt.loss_name = trial.suggest_categorical('loss_name', [  'SpikeF1scoreLoss', 'MSELoss'])

    new_opt.lif_beta = trial.suggest_float('lif_beta', 0, 1)
    scale = 10
    scale = 3.618    
    new_opt.alpha_surrogate = trial.suggest_float('alpha_surrogate', new_opt.alpha_surrogate / scale, new_opt.alpha_surrogate * scale, log=True)
    new_opt.base_lr = trial.suggest_float('base_lr', new_opt.base_lr / scale, new_opt.base_lr * scale, log=True)
    new_opt.final_lr = trial.suggest_float('final_lr', new_opt.final_lr / scale, new_opt.final_lr * scale, log=True)
    new_opt.delta1 = trial.suggest_float('delta1', new_opt.delta1 / scale, min((new_opt.delta1 * scale, .99)), log=True)
    new_opt.delta2 = trial.suggest_float('delta2', new_opt.delta2 / scale, min((new_opt.delta2 * scale, .99)), log=True)
    new_opt.weight_decay = trial.suggest_float('weight_decay', new_opt.weight_decay / scale, new_opt.weight_decay * scale, log=True)
    new_opt.dropout = trial.suggest_float('dropout', 0, 1)
    new_opt.init_gain = trial.suggest_float('init_gain', new_opt.init_gain / scale, new_opt.init_gain * scale, log=True)

    try:
        hd = HD_SNN(new_opt)
        hd.update_weight()
        hd.learn_model(verbose=False)
        
        with torch.no_grad():
            # draw causes (SMs) - hidden to the observer
            input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay)
            _, _, spikes = hd.forward_pass(input_spikes)
            spikes_ = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
            target_ = hd.target[:, :, hd.opt.num_delay:]
            loss_val = loss_fn(spikes_, target_)

        return loss_val.item()
    except:
        return 1.
        
N_optuna_scan = 1000
N_optuna_scan = 100

# 3. Create a study object and optimize the objective function.
study = optuna.create_study(
    storage=f"sqlite:///{str(optuna_filename)}",
    sampler=optuna.samplers.TPESampler(multivariate=True, warn_independent_sampling=False),
    direction="minimize",
    load_if_exists=True,
    study_name=opt.datetag,
)

if len(study.trials)>0:
    print("So far, Best params: ", study.best_params, end=', ')
    print("Best value: ", study.best_value)

print(
    f"Starting optimization on {max(N_optuna_scan - len(study.trials), 0)} trials with {len(study.trials)} existing trials."
)
study.optimize(objective, n_trials=max((N_optuna_scan - len(study.trials), 0)), n_jobs=1, show_progress_bar=True)


In [ ]:
study = optuna.load_study(storage=f"sqlite:///{str(optuna_filename)}", study_name=opt.datetag)

In [ ]:
print(50 * "-.")
print("Best params: ", study.best_params)
print("Best value: ", study.best_value)
# print("Best Trial params: ", study.best_trial.params)

In [ ]:
torch.linspace(-0.5, 0.5, 2)

In [ ]:
import optuna.visualization.matplotlib as vis
vis.plot_param_importances(study)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import numpy as np
import pandas as pd

# Charger l'étude
params = sorted({k for t in study.trials for k in t.params})

fig, axes = plt.subplots(len(params), 1, figsize=(15, 8 * len(params)), sharey=True)

for ax, pname in zip(axes, params):
    xs = [t.params[pname] for t in study.trials if pname in t.params]
    ys = [t.value for t in study.trials if pname in t.params]

    # Créer un DataFrame pour faciliter la manipulation
    df = pd.DataFrame({"x": xs, "y": ys})
    # Déterminer si le paramètre est numérique (et non catégoriel)
    is_numeric = pd.api.types.is_numeric_dtype(df["x"])
    if is_numeric:
        # Si numérique, vérifier s'il y a assez de valeurs distinctes pour LOWESS
        if len(df["x"].unique()) > 1:
            sns.regplot(
                x="x", y="y", data=df, ax=ax, lowess=True, color="red",
                line_kws={"lw": 2, "label": "Tendance (LOWESS)"}, scatter=False
            )
    else:
        # Pour les catégories, convertir en chaîne et afficher un boxplot
        df["x"] = df["x"].astype(str)
    #     sns.boxplot(x="x", y="y", data=df, ax=ax, color="lightblue", width=0.5)

    # Toujours afficher les points bruts
    ax.scatter(df["x"], df["y"], s=20, alpha=0.7, color="blue", label="Trials")

    ax.set_xlabel(pname)
    ax.set_ylabel("Objective")
    ax.set_yscale("log")
    if pname not in ['learn_beta', 'learn_threshold', 'optimizer', 'reset_mechanism'] :
        ax.set_xscale("log")

    # ax.legend()

plt.tight_layout()